# 1 - Data cleaning
We retrieve the [Kaggle dataset](https://www.kaggle.com/datasets/Cornell-University/arxiv/data) using `kagglehub`, and clean it up.  We first remove all unnecessary data, to make the dataset more manageable, then extract the date of first posting, and finally deal with the categories that over time changed names, using the list of current categories (`data/arxiv-categories.json`).

##### Starting point
- the arXiv metadata from Kaggle: a `json` file whose entries are individual preprints with the following fields
    - `id` (string) arXiv identifier
    - `submitter` (string)
    - `authors` (string)
    - `title` (string)
    - `comments` (string)
    - `journal-ref` (string)
    - `doi` (string) [Digital Object Identifier](https://www.doi.org/)
    - `report-no` (string) report number
    - `categories` (string)
    - `license` (string)
    - `abstract` (string)
    - `versions` list: each entry is a dictionary of `version` (string - number of version) and `created` (string - date of creation of the version)
    - `update-date` (string) date of last update
    - `author-parsed` list: each entry is a list of the names of the authors (last, first, middle)
- the list of all current categories: `data/arxiv-categories.json`

##### End goal
A cleaned metadata file `data/arxiv-metadata-cleaned.parquet` whose inconsistencies issues have been resolved.

## The code

In [1]:
import pandas as pd
import kagglehub

### Pre-cleaning
We begin by removing the unused data, to make the dataset lighter.  First, we load the dataset using `kagglehub`.

In [ ]:
arxiv_dataset = kagglehub.load_dataset(
    kagglehub.KaggleDatasetAdapter.PANDAS,
    "Cornell-University/arxiv",
    "arxiv-metadata-oai-snapshot.json",
    pandas_kwargs={"lines": True},
)
arxiv_dataset.head()

/var/folders/bc/nlpbp2z15m5bs2w20xdp390ddkpfsf/T/ipykernel_30844/1609841259.py:1: DeprecationWarning: load_dataset is deprecated and will be removed in a future version.
  arxiv_dataset = kagglehub.load_dataset(


,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,..."
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"
2,0704.0003,Hongjun Pan,Hongjun Pan,The evolution of the Earth-Moon system based o...,"23 pages, 3 figures",None,None,None,physics.gen-ph,None,The evolution of Earth-Moon system is descri...,"[{'version': 'v1', 'created': 'Sun, 1 Apr 2007...",2008-01-13,"[[Pan, Hongjun, ]]"
3,0704.0004,David Callan,David Callan,A determinant of Stirling cycle numbers counts...,11 pages,None,None,None,math.CO,None,We show that a determinant of Stirling cycle...,"[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2007-05-23,"[[Callan, David, ]]"
4,0704.0005,Alberto Torchinsky,Wael Abu-Shammala and Alberto Torchinsky,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,None,"Illinois J. Math. 52 (2008) no.2, 681-689",None,None,math.CA math.FA,None,In this paper we show how to compute the $\L...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2013-10-15,"[[Abu-Shammala, Wael, ], [Torchinsky, Alberto, ]]"


Then we extract the `id`, `versions`, and `categories` columns and save the new dataset to `data/arxiv-metadata-id-versions-categories.parquet`.

In [5]:
arxiv_stripped = arxiv_dataset[["id", "versions", "categories"]]
arxiv_stripped.to_parquet("../data/arxiv-metadata-id-versions-categories.parquet")

### Cleaning

In [6]:
arxiv_stripped = pd.read_parquet(
    "../data/arxiv-metadata-id-versions-categories.parquet"
)

#### Date of v1 extraction
We use the date of v1 as reference points.  This is for several reasons:
- `update_date` is not a reliable source since all papers got updated in May 2007, and thus does not record older dates
- v1 is easy to extract from each entry in versions: `version[0]['created']`

We begin by defining a function `date_extractor` that takes the list of versions and returns a `pd.Timestamp` containing the first date appearing in the list (i.e. the date of v1), shifted according to the [arXiv announcement schedule](https://info.arxiv.org/help/availability.html#announcement-schedule):
- if v1 is before 18:00:00 UTC, then the announcement date (`date`) is the next business day,
- if v1 is after 18:00:00 UTC, then the announcement date is two business days after.

Note that this does not take into account DST shifts in announcement schedules.  We ignore this for the moment because it probably has a relatively small effect, but it introduces a bias.

In [7]:
def date_extractor(versions):
    timestamp = pd.Timestamp(versions[0]["created"])
    if timestamp.hour < 18:
        return pd.Timestamp(
            timestamp.year, timestamp.month, timestamp.day
        ) + pd.offsets.BusinessDay(1)
    else:
        return pd.Timestamp(
            timestamp.year, timestamp.month, timestamp.day
        ) + pd.offsets.BusinessDay(2)

Next, we create a new `date` column by applying `date_extractor` to the `versions` column.

In [8]:
arxiv_stripped["date"] = arxiv_stripped["versions"].apply(date_extractor)

Finally, we drop the `versions` column and save the new `arxiv_metadata` dataset to `data/arxiv-id-date-categories.parquet`.

In [9]:
arxiv_metadata = arxiv_stripped.drop(columns=["versions"])
arxiv_metadata.to_parquet("../data/arxiv-metadata-id-date-categories.parquet")

This is how our dataset looks now.

In [10]:
arxiv_metadata

,id,categories,date
0,0704.0001,hep-ph,2007-04-04
1,0704.0002,math.CO cs.CG,2007-04-02
2,0704.0003,physics.gen-ph,2007-04-03
3,0704.0004,math.CO,2007-04-02
4,0704.0005,math.CA math.FA,2007-04-04
...,...,...,...
2765255,supr-con/9608008,supr-con cond-mat.supr-con,1996-08-27
2765256,supr-con/9609001,supr-con cond-mat.supr-con,1996-09-02
2765257,supr-con/9609002,supr-con cond-mat.supr-con,1996-09-04
2765258,supr-con/9609003,supr-con cond-mat.supr-con,1996-09-19


#### Categories cleaning

First we import the list of current arXiv category tags and store it in the list `arxiv_categories`.

In [11]:
import json

with open("../data/arxiv-categories.json", "r") as f:
    arxiv_categories_descriptions = json.load(f)

arxiv_categories = [cat["tag"] for cat in arxiv_categories_descriptions]

Now we import the stripped data as `arxiv_metadata`.

In [12]:
arxiv_metadata = pd.read_parquet("../data/arxiv-metadata-id-date-categories.parquet")

Categories are written as a simple string listing all categories separated by white spaces: we split them into a list of words (each word is one category).

In [13]:
arxiv_metadata["categories"] = arxiv_metadata["categories"].apply((lambda s: s.split()))

The arXiv categories changed over the years: we find all categories that are not the current ones and store them in the set `missing_categories`.

In [14]:
missing_categories = set()

for index, row in arxiv_metadata.iterrows():
    for category in row["categories"]:
        if category not in arxiv_categories:
            missing_categories.add(category)

print(missing_categories)

{'dg-ga', 'bayes-an', 'acc-phys', 'q-bio', 'atom-ph', 'patt-sol', 'funct-an', 'cond-mat', 'mtrl-th', 'ao-sci', 'astro-ph', 'alg-geom', 'comp-gas', 'supr-con', 'chao-dyn', 'solv-int', 'plasm-ph', 'cmp-lg', 'chem-ph', 'q-alg', 'adap-org'}


We need to decide what to do for each of the missing categories.  The most reasonable choice to me seems to find the closest matching current category and replace each missing category with that.

| Old        |  New                | To add? |
| ---------- | ------------------- | :-----: |
| `mtrl-th`  | `cond-mat.mtrl-sci` |         |
| `q-bio`    | ---                 | X       |
| `acc-phys` | `physics.acc-ph`    |         |
| `dg-ga`    | `math.DG`           |         |
| `cond-mat` | ---                 | X       |
| `chem-ph`  | `physics.chem-ph`   |         |
| `astro-ph` | ---                 | X       |
| `comp-gas` | `nlin.CG`           |         |
| `funct-an` | `math.FA`           |         |
| `patt-sol` | `nlin.PS`           |         |
| `solv-int` | `nlin.SI`           |         |
| `alg-geom` | `math.AG`           |         |
| `adap-org` | `nlin.AO`           |         |
| `supr-con` | `cond-mat.supr-con` |         |
| `plasm-ph` | `physics.plasm-ph`  |         |
| `chao-dyn` | `nlin.CD`           |         |
| `bayes-an` | `physics.data-an`   |         |
| `q-alg`    | `math.QA`           |         |
| `ao-sci`   | `physics.ao-ph`     |         |
| `atom-ph`  | `physics.atom-ph`   |         |
| `cmp-lg`   | `cs.CL`             |         |

The categories `q-bio`, `cond-mat`, and `astro-ph` have been over the years split into subcategories.  Hence, some preprints are classified into what are now meta-categories.  We add these three categories, and we will use them only for those preprints dating to before the splitting.

The goal now is to go through `arxiv_metadata` again and replace the missing categories with the new ones.  We start by creating a dictionary `cat_dictionary` to map old categories to new categories, and a function `translate` to translate a list of categories to the new ones using a given dictionary (removing duplicates).

In [15]:
cat_dictionary = {
    "alg-geom": "math.AG",
    "dg-ga": "math.DG",
    "chem-ph": "physics.chem-ph",
    "plasm-ph": "physics.plasm-ph",
    "ao-sci": "physics.ao-ph",
    "mtrl-th": "cond-mat.mtrl-sci",
    "funct-an": "math.FA",
    "comp-gas": "nlin.CG",
    "q-alg": "math.QA",
    "acc-phys": "physics.acc-ph",
    "atom-ph": "physics.atom-ph",
    "supr-con": "cond-mat.supr-con",
    "chao-dyn": "nlin.CD",
    "bayes-an": "physics.data-an",
    "cmp-lg": "cs.CL",
    "patt-sol": "nlin.PS",
    "adap-org": "nlin.AO",
    "solv-int": "nlin.SI",
}


def translate(categories: list, dictionary: dict) -> list:
    return sorted(
        set([dictionary[cat] if cat in dictionary else cat for cat in categories])
    )

Then we traverse the `categories` column in `arxiv_metadata` and use the dictionary `cat_translator` to update categories.

In [16]:
arxiv_metadata["categories"] = arxiv_metadata["categories"].apply(
    lambda x: translate(x, cat_dictionary)
)

Here is how the cleaned dataset looks like.

In [17]:
arxiv_metadata

,id,categories,date
0,0704.0001,[hep-ph],2007-04-04
1,0704.0002,"[cs.CG, math.CO]",2007-04-02
2,0704.0003,[physics.gen-ph],2007-04-03
3,0704.0004,[math.CO],2007-04-02
4,0704.0005,"[math.CA, math.FA]",2007-04-04
...,...,...,...
2765255,supr-con/9608008,[cond-mat.supr-con],1996-08-27
2765256,supr-con/9609001,[cond-mat.supr-con],1996-09-02
2765257,supr-con/9609002,[cond-mat.supr-con],1996-09-04
2765258,supr-con/9609003,[cond-mat.supr-con],1996-09-19


We save the new cleaned file to `data/arxiv-metadata-cleaned.parquet`.

In [18]:
arxiv_metadata.to_parquet("../data/arxiv-metadata-cleaned.parquet")